# Corrective RAG (CRAG)

Grade retrieval quality; refine good evidence, and fall back to web search when it is poor.

**Extra requirements:** `pip install duckduckgo-search`

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
def grade(query, chunk):
    verdict = llm(f"Is the passage relevant to the question? Reply exactly one word: correct, ambiguous or incorrect.\n\nQuestion: {query}\n\nPassage: {chunk}").lower()
    return next((v for v in ("incorrect", "ambiguous", "correct") if v in verdict), "ambiguous")

def refine(query, chunk):
    """Knowledge refinement: keep only the useful sentences."""
    return llm(f"Copy only the sentences relevant to the question, verbatim.\n\nQuestion: {query}\n\nPassage: {chunk}")

def web_search(query, n=3):
    try:
        from duckduckgo_search import DDGS
        return [r["body"] for r in DDGS().text(query, max_results=n)]
    except Exception as e:
        print("web search unavailable:", e)
        return []

In [ ]:
def crag(query):
    hits = store.search(query, 4)
    grades = [grade(query, h["text"]) for h in hits]
    good = [refine(query, h["text"]) for h, g in zip(hits, grades) if g == "correct"]
    if good:
        action, contexts = "use retrieved", good
    elif "ambiguous" in grades:
        rewritten = llm(f"Rewrite as a short web search query: {query}")
        action, contexts = "retrieved + web", [h["text"] for h, g in zip(hits, grades) if g == "ambiguous"] + web_search(rewritten)
    else:
        action, contexts = "web search only", web_search(llm(f"Rewrite as a short web search query: {query}"))
    return action, grades, (answer(query, contexts) if contexts else "No evidence found.")

for q in ["What is HNSW?", "Who is the current prime minister of Japan?"]:
    action, grades, response = crag(q)
    print(f"Q: {q}\n  grades={grades}\n  action={action}\n  A: {response[:300]}\n")